# Thrail App Recommendation System (TARS) Scratchpad & Prototyping Notebook
This scratchpad demonstrates the official TARS mathematical modules and execution workflow:
1. **Module 1**: Gower's Distance Similarity Engine (Mixed Data Types: Numerical, Ordinal, Nominal)
2. **Module 2**: User Base Profiling, Dynamic Alpha ($\alpha$) Calibration & Incremental Profile Recalibration
3. **Module 3**: User-User Collaborative Filtering & Hybrid Score Fusion ($R_{UTx}$)
4. **Module 4**: Upper Triangular Matrix Optimization & Self-Similarity Bypass
5. **Module 5**: End-to-End Recommendation Pipeline Demonstration

In [10]:
import sys
import os
import numpy as np
import pandas as pd

# Import TARS core modules
from gower_engine import GowerSimilarityEngine, compute_gower_similarity_single, DEFAULT_FEATURE_CONFIG
from recommender import HybridRecommender, calibrate_alpha, recalibrate_user_profile, BASE_PROFILES

## 1. Load Datasets
Loading simulated Cloud Firestore dataset expanded to **10 candidate trails** and **50 active users** (`user_001` to `user_050`).

In [11]:
trails_df = pd.read_csv('data/trails_mock.csv')
ratings_df = pd.read_csv('data/user_ratings_mock.csv')

print(f"Loaded {len(trails_df)} candidate trails and {len(ratings_df)} historical user reviews from {ratings_df['user_id'].nunique()} unique users.")
trails_df[['id', 'name', 'province', 'difficulty_lascoRating', 'difficulty_length', 'difficulty_gain', 'difficulty_hours']]

Loaded 10 candidate trails and 54 historical user reviews from 50 unique users.


,id,name,province,difficulty_lascoRating,difficulty_length,difficulty_gain,difficulty_hours
0,trail_001,Mt. Daraitan,Rizal,4,8.0,600,4.5
1,trail_002,Mt. Batulao,Batangas,3,12.0,400,4.0
2,trail_003,Mt. Maculot,Batangas,4,6.0,500,3.5
3,trail_004,Mt. Makiling,Laguna,6,16.0,900,7.0
4,trail_005,Mt. Pico de Loro,Cavite,3,9.5,350,4.0
5,trail_006,Mt. Talamitam,Batangas,3,7.5,300,3.0
6,trail_007,Mt. Gulugod Baboy,Batangas,2,5.0,250,2.5
7,trail_008,Mt. Cristobal,Laguna,7,14.0,1100,8.0
8,trail_009,Mt. Sembrano,Rizal,5,11.0,650,5.0
9,trail_010,Mt. Kalisungan,Laguna,4,8.5,480,4.0


## 2. Module 1: Gower's Distance Similarity Engine
Demonstrating attribute-level similarity calculation $S_j(A_j, B_j)$ across mixed data types:
- **Numerical**: $S_j = 1 - \frac{|A_j - B_j|}{\text{Range}_j}$
- **Ordinal**: $S_j = 1 - |\text{norm}(A_j) - \text{norm}(B_j)|$
- **Nominal**: $S_j = 1$ if match else $0$

In [12]:
gower_engine = GowerSimilarityEngine()

# Sample User Profile Vector vs Trail Vector
user_profile_beginner = {
    'difficulty_gain': 200.0,
    'difficulty_length': 4.0,
    'difficulty_lascoRating': 3.0,
    'province': 'rizal'
}

trail_daraitan = {
    'difficulty_gain': 600.0,
    'difficulty_length': 8.0,
    'difficulty_lascoRating': 4.0,
    'province': 'rizal'
}

sim_score = compute_gower_similarity_single(user_profile_beginner, trail_daraitan, DEFAULT_FEATURE_CONFIG)
print(f"Gower Similarity between Beginner User Profile & Mt. Daraitan: {sim_score:.4f}")

Gower Similarity between Beginner User Profile & Mt. Daraitan: 0.8854


## 3. Module 2: User Base Profiling, Dynamic Alpha Calibration & Profile Recalibration
- Base profiles: `Beginner`, `Regular`, `Experienced`
- Dynamic $\alpha$: $1.0$ (0 hikes) $\rightarrow 0.75$ (1-2 hikes) $\rightarrow 0.50$ (3+ hikes)
- Profile Error Correction: Updates user capacity via Exponential Moving Average (EMA) and post-hike survey rating shift factor $\delta$.

In [13]:
# A. Dynamic Alpha Calibration Test
print("Alpha for 0 hikes:", calibrate_alpha(0))
print("Alpha for 2 hikes:", calibrate_alpha(2))
print("Alpha for 5 hikes:", calibrate_alpha(5))

# B. Incremental Profile Recalibration with GPS Log & Survey Rating ('Easy')
base_profile = BASE_PROFILES['beginner'].copy()
gps_log = {'distance': 10000.0, 'duration': 14400.0, 'elevation': 600.0}

recalibrated_profile = recalibrate_user_profile(base_profile, gps_log, perceived_difficulty='Easy')
print("\nOriginal Beginner Profile:", base_profile)
print("Recalibrated User Profile:", recalibrated_profile)

Alpha for 0 hikes: 1.0
Alpha for 2 hikes: 0.75
Alpha for 5 hikes: 0.5

Original Beginner Profile: {'difficulty_lascoRating': 3.0, 'difficulty_length': 4.0, 'difficulty_gain': 200.0, 'difficulty_slope': 8.0, 'difficulty_hours': 2.0}
Recalibrated User Profile: {'difficulty_lascoRating': 3.3, 'difficulty_length': 6.1, 'difficulty_gain': 330.0, 'difficulty_slope': 8.0, 'difficulty_hours': 2.8}


## 4. Module 4: Upper Triangular Symmetric Matrix Optimization
Demonstrating upper triangular calculation ($S(A,B) = S(B,A)$) and self-similarity bypass ($S(A,A) = 1.0$), which cuts pairwise operations by **>50%**.

In [14]:
# Compute pairwise matrix on trails dataframe (10 x 10)
matrix, stats = gower_engine.compute_pairwise_matrix(trails_df, is_symmetric=True)

print("Pairwise Trail Similarity Matrix Shape:", matrix.shape)
print("Optimization Performance Stats:", stats)
print(f"Efficiency Gain: {stats['efficiency_gain_pct']}% operations skipped!")

Pairwise Trail Similarity Matrix Shape: (10, 10)
Optimization Performance Stats: {'total_pairs': 100, 'computed_pairs': 45, 'skipped_pairs': 55, 'efficiency_gain_pct': 55.0}
Efficiency Gain: 55.0% operations skipped!


## 5. Module 3 & 5: Full End-to-End Hybrid Recommendation Demonstration across 50 Peer Users
Executes the full hybrid pipeline blending Content-Based Gower similarity and User-User Collaborative Filtering across 50 active peer users.

In [15]:
recommender = HybridRecommender('data/trails_mock.csv', 'data/user_ratings_mock.csv')

user_preferences = {
    'experience': 'Beginner',
    'province': ['Batangas', 'Rizal'],
    'hike_length': ['1-3 Hour(s)'],
    'hiked': True,
    'location': []
}

# Scenario 1: Cold Start User (0 completed hikes -> alpha = 1.0)
cold_recs = recommender.get_hybrid_recommendations(
    user_id='user_cold_start',
    preferences=user_preferences,
    top_k=5,
    completed_hikes=0
)
print("=== COLD START RECOMMENDATIONS (alpha = 1.0) ===")
print(pd.DataFrame(cold_recs['recommendations']))

# Scenario 2: Active User (3 completed hikes -> alpha = 0.50, matching against 50 peer users)
active_recs = recommender.get_hybrid_recommendations(
    user_id='user_001',
    preferences=user_preferences,
    top_k=5,
    completed_hikes=3,
    gps_summary={'distance': 8200.0, 'duration': 16200.0, 'elevation': 610.0},
    last_perceived_difficulty='Easy'
)
print("\n=== ACTIVE USER HYBRID RECOMMENDATIONS (alpha = 0.50) ===")
print(pd.DataFrame(active_recs['recommendations']))

=== COLD START RECOMMENDATIONS (alpha = 1.0) ===
    trail_id         trail_name  match_score  cbf_score  cf_score  \
0  trail_007  Mt. Gulugod Baboy       0.8559     0.8559    0.8559   
1  trail_002        Mt. Batulao       0.8454     0.8454    0.8454   
2  trail_006      Mt. Talamitam       0.8033     0.8033    0.8033   
3  trail_005   Mt. Pico de Loro       0.7897     0.7897    0.7897   
4  trail_010     Mt. Kalisungan       0.7726     0.7726    0.7726   

                                              reason  
0  Gower Content Match for Batangas based on your...  
1  Gower Content Match for Batangas based on your...  
2  Gower Content Match for Batangas based on your...  
3  Gower Content Match for Cavite based on your i...  
4  Gower Content Match for Laguna based on your i...  

=== ACTIVE USER HYBRID RECOMMENDATIONS (alpha = 0.50) ===
    trail_id         trail_name  match_score  cbf_score  cf_score  \
0  trail_007  Mt. Gulugod Baboy       0.9125     0.8538    0.9713   
1  trail_